# IPL Toss Match Outcome Prediction

**Machine Learning Classification Project**

This notebook analyses IPL match data from **2008–2024** and builds a classification model to predict whether the **toss winner also wins the match**.

> **Important:** This is a **post-toss prediction problem** because toss-related information is used as input. It is not a pre-toss match-winner prediction.

## Project Workflow
1. Data loading
2. Data understanding and cleaning
3. Exploratory Data Analysis (EDA)
4. Feature selection
5. One-hot encoding
6. Train/test split
7. Logistic Regression
8. Decision Tree
9. Model evaluation
10. Model saving/loading
11. New-match prediction

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt

## 1. Load Dataset

In [ ]:
df =pd.read_csv('matches_2008-2024.csv')


## 2. Data Understanding & Cleaning

In [ ]:
# df.isnull().sum()
df.duplicated().sum()

In [ ]:
# null value fill 
df['city'].isnull().sum()
df['city'] = df['city'].ffill()

In [ ]:

df.drop_duplicates(inplace= True)

In [ ]:
df.dtypes

In [ ]:
df['city'].unique()
df['super_over'].unique()
df['method'].unique()

In [ ]:
df.nunique()
df['super_over'].nunique()
df['method'].nunique()

In [ ]:
df.count()

In [ ]:
df.info()


In [ ]:
df['super_over'] = df['super_over'].map({
    'N':False,
    'Y':True})
 


In [ ]:
df['method'] = df['method'].fillna('no method')


In [ ]:
df.isnull().sum()

In [ ]:
df[df['winner'].isnull()][["season","team1","team2","result","winner"]]

In [ ]:
df[df['result_margin'].isnull()][["team1","team2","winner","result","result_margin"]]

In [ ]:
df[df['player_of_match'].isnull()][["team1","team2","winner","result","player_of_match"]]

In [ ]:
df[df["target_runs"].isnull()][
    ["team1", "team2", "winner", "result", "target_runs", "target_overs"]
]

In [ ]:
df[df['result']=='tie'][["team1","team2","winner","result","player_of_match"]]

In [ ]:
df[df["result"] == "no result"][["team1", "team2", "winner", "result","player_of_match","target_runs","target_overs"]]

In [ ]:
df.duplicated().sum()

In [ ]:
df["result"].value_counts(dropna = False)

In [ ]:
df[df["result_margin"].isnull()]["result"].value_counts()

In [ ]:
df["toss_decision"].value_counts()


In [ ]:
df["result"].value_counts()

In [ ]:
df["match_type"].value_counts()

In [ ]:
df["team1"].value_counts()

In [ ]:
df["team2"].value_counts()

In [ ]:
df["toss_winner"].value_counts()

In [ ]:
df["winner"].value_counts()

In [ ]:
team_name_map = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Kings XI Punjab": "Punjab Kings",
    "Delhi Daredevils": "Delhi Capitals",
    "Rising Pune Supergiants": "Rising Pune Supergiant"
}

In [ ]:
col = ["team1","team2","winner","toss_winner"]
for n in col:
    df[col]= df[col].replace(team_name_map)


In [ ]:
df["winner"].value_counts(dropna = False)

In [ ]:
df["team1"].value_counts()

df["team2"].value_counts()

df["toss_winner"].value_counts()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
df["winner"].value_counts().head(-14)

In [ ]:
top_10 = df["winner"].value_counts().head(10)

plt.barh(top_10.index,top_10.values)
# top_10.values

plt.xlabel("Number of win")
plt.ylabel("Team")
plt.show()


In [ ]:
 df["toss_match_result"] = df["toss_winner"] == df["winner"]

In [ ]:
truth=df["toss_match_result"].value_counts()[True]
total_win = df["toss_match_result"].value_counts().sum()
falses=  total_win-truth

In [ ]:
percentage = (truth*100)/total_win
percentage

In [ ]:
#upgrade 
import matplotlib.pyplot as plt

plt.figure(figsize=(4,4))
counts = df["toss_match_result"].value_counts()

# counts.values = [554 541]

plt.bar(["Toss Winner Won", "Toss Winner Lost"], counts.values,0.25)

plt.xlabel("Result")
plt.ylabel("Number of Matches")
plt.title("Toss Winner vs Match Winner")


In [ ]:
total_dec = df["toss_decision"].value_counts().sum()

In [ ]:
pd.crosstab(df["toss_decision"], df["result"])

In [ ]:
from matplotlib.ticker import PercentFormatter
plt.figure(figsize = (5,5))
plt.bar(percentage.index, percentage.values)
plt.title("Toss Decision: Batting vs Fielding")
plt.xlabel("Toss Decision")
plt.ylabel("Percentage of matches (%)")
plt.show()

In [ ]:
stadium_name_map={ 
    "Feroz Shah Kotla": "Arun Jaitley Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",

    "Wankhede Stadium, Mumbai": "Wankhede Stadium",

    "Eden Gardens, Kolkata": "Eden Gardens",

    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium",

    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium",

    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",

    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium",
    
    "Punjab Cricket Association Stadium, Mohali":
        "Punjab Cricket Association IS Bindra Stadium, Mohali",

    "Punjab Cricket Association IS Bindra Stadium":
        "Punjab Cricket Association IS Bindra Stadium, Mohali",

    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh":
        "Punjab Cricket Association IS Bindra Stadium, Mohali"
}
    


In [ ]:
df["venue"] = df["venue"].replace(stadium_name_map)
df["venue"].unique()

In [ ]:
venue_matches  = df["venue"].value_counts().head(10)
venue_matches.head(10)

In [ ]:
df.groupby("toss_decision")["toss_match_result"].value_counts()

In [ ]:
winn_decision  = df.groupby("toss_decision")["toss_match_result"].mean()*100

winn_decision

In [ ]:
plt.figure(figsize=(4,6))
plt.bar(winn_decision.index,winn_decision.values)
plt.xlabel("Toss Decision")
plt.ylabel("Match Win Percentage (%)")
plt.title("Toss Decision vs Match Win Percentage")
plt.show()

In [ ]:
pd.crosstab(df["venue"],df["result"])

In [ ]:
pd.crosstab(df["venue"], df["result"], normalize="index").mul(100).round(2) 

In [ ]:
result_bywicket = pd.crosstab(df["venue"], df["result"], normalize="index").mul(100).round(2) .sort_values("wickets",ascending = False)
result_bywicket.head(25)

In [ ]:
result_byrun = pd.crosstab(df["venue"], df["result"], normalize="index").mul(100).round(2) .sort_values("runs",ascending = False)


result_byrun.head(20)

In [ ]:
top_10_players = df["player_of_match"].value_counts().head(10)

In [ ]:
sns.barplot(x=top_10_players.values,y=top_10_players.index,orient="h")
plt.title("Top 10 Players by Player of the Match Awards")
plt.ylabel("players")
plt.xlabel("number of palyer of the match ")
# for i, value in enumerate(top_10_players.values):
#     plt.text(value + 0.2, i, str(value), va="center")
plt.show()



In [ ]:
season_matches  = df["season"].value_counts().sort_index()

In [ ]:
sns.lineplot(x=season_matches.index,y=season_matches.values,marker ='o')
plt.title("season matches")
plt.xlabel("years")
plt.ylabel("number of matches")
plt.show()

In [ ]:
df[df["match_type"]=="Final"][["season","winner"]].sort_values("season")

In [ ]:
champion_count = df[df["match_type"]=="Final"]["winner"].value_counts()
champion_count

In [ ]:
sns.barplot(y = champion_count.index,x = champion_count.values,orient = "h")
plt.ylabel("champion team")
plt.xlabel("number of championship")
plt.title("champion team count")
plt.show()

In [ ]:
# Har season mein sabse zyada Player of the Match awards kis player ko mile?


# please comment out it 
season_player_awards = (df.groupby(["season", "player_of_match"]).size().reset_index(name = "awards"))
# season_player_awards

In [ ]:
#har season ka top Player of the Match 
season_player_awards.loc[season_player_awards.groupby("season")["awards"].idxmax()].sort_values("season")

In [ ]:
df["toss_winner"].value_counts().head(10)

In [ ]:
# Har IPL season mein kitne matches D/L method se decide hue.
dls_matchs  = df[df["method"] == "D/L"]["season"].value_counts().sort_index()
dls_matchs

In [ ]:
# result :-Matches ka result runs se aaya ya wickets se?

# bar chart 

result_base = df["result"].value_counts()
result_base

In [ ]:
# bar chart result
plt.figure(figsize=(8,8))
sns.barplot(x=result_base.values,y=result_base.index)

plt.title("decision of result")
plt.ylabel("Function")
plt.xlabel("number winnig")
plt.xticks(range(0, 601, 25))
plt.show()

In [ ]:
# df.groupby(["season", "result"]).size().unstack(fill_value=0)
df.groupby(["season","result"]).size().unstack(fill_value=0)

In [ ]:
pd.crosstab(df["toss_decision"], df["result"])

In [ ]:
pd.crosstab(df["toss_decision"], df["result"], normalize="index") * 100

## 4. Machine Learning Preparation

In [ ]:
df["toss_match_result"].value_counts()

In [ ]:
y = df["toss_match_result"]

In [ ]:
X = df[["season", "city", "match_type", "venue", "team1", "team2", "toss_winner", "toss_decision"]]
X.head(3)


In [ ]:
categorical_cols = ["city", "match_type", "venue", "team1", "team2", "toss_winner", "toss_decision"]

In [ ]:
categorical_cols = ["city", "match_type", "venue", "team1", "team2", "toss_winner", "toss_decision"]

X_encoded = pd.get_dummies(X, columns=categorical_cols, dtype=int)

In [ ]:
X_encoded.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

## 5. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
#logistic-regression
model = LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
y_test.value_counts(normalize=True)*100

## 6. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model

In [ ]:
dt_model.fit(X_train,y_train)

In [ ]:
dt_pred = dt_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, dt_pred)*100

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, dt_pred)

In [ ]:
from sklearn.metrics import classification_report

print((classification_report(y_test, dt_pred)))


## 7. Tuned Decision Tree

In [ ]:
dt_model_9 = DecisionTreeClassifier(max_depth=9, random_state=42)

In [ ]:
dt_model_9.fit(X_train, y_train)

In [ ]:
dt_pred_9 = dt_model_9.predict(X_test)

In [ ]:
accuracy_score(y_test, dt_pred_9) * 100

In [ ]:
confusion_matrix(y_test, dt_pred_9)

In [ ]:
# classification details 
print(classification_report(y_test, dt_pred_9))

## 8. Final Model Evaluation

In [ ]:
train_accuracy = accuracy_score(
    y_train,
    dt_model_9.predict(X_train)
) * 100

test_accuracy = accuracy_score(
    y_test,
    dt_pred_9
) * 100

print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)

## 9. Save the Model

In [ ]:
import joblib

joblib.dump(dt_model_9, "ipl_toss_match_model.pkl")

## 10. Load and Verify the Saved Model

In [ ]:
loaded_model = joblib.load("ipl_toss_match_model.pkl")
loaded_model

In [ ]:
loaded_pred = loaded_model.predict(X_test)

accuracy_score(y_test, loaded_pred) * 100

## 11. New Match Prediction

In [ ]:
new_match = pd.DataFrame({
    "season": [2024],
    "city": ["Mumbai"],
    "match_type": ["League"],
    "venue": ["Wankhede Stadium"],
    "team1": ["Mumbai Indians"],
    "team2": ["Chennai Super Kings"],
    "toss_winner": ["Mumbai Indians"],
    "toss_decision": ["field"]
})

new_match

In [ ]:
new_match_encoded = pd.get_dummies(
    new_match,
    columns=categorical_cols,
    dtype=int
)

new_match_encoded

In [ ]:
new_match_encoded = new_match_encoded.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [ ]:
new_prediction = loaded_model.predict(new_match_encoded)

new_prediction

In [ ]:
loaded_model.predict_proba(new_match_encoded)

## 12. Conclusion

In [ ]:
# loaded_model.classes_

## Final Results

The selected Decision Tree uses `max_depth=9`.

- **Training accuracy:** 65.64%
- **Testing accuracy:** 57.53%
- **Majority-class baseline:** approximately 51.14%

The model performs somewhat better than the baseline, but the test accuracy is still modest. This makes the project useful as a practical ML learning/portfolio project rather than a production-grade prediction system.

### Limitations

- The prediction is made **after the toss**.
- No-result matches do not have a match winner and therefore require careful interpretation when constructing the target.
- Decision-tree `predict_proba()` values should not be treated as guaranteed real-world probabilities.
